# Process Data

In [11]:
###### Input variables ######

INPUT_PDB_DIR = "/home/psh/data/CAD6_GKR01I/pdb"

CDR_SEQS = {
    "h1": "GYTFTRNFMH",
    "h2": "WIYPGDGE",
    "h3": "GVYGGFAGGYFDF",
    "l1": "KASQNIYKNLA",
    "l2": "DANTLQT",
    "l3": "QQYYSGWA"
}


In [12]:
###### chothia numbering ######
import os
from data.ab_metrics import renumber_pdb_wt_constant

chothia_dir = INPUT_PDB_DIR.replace('pdb', 'pdb_chothia')
os.makedirs(chothia_dir, exist_ok=True)

for file in os.listdir(INPUT_PDB_DIR):
    filepath = os.path.join(INPUT_PDB_DIR, file)
    outpath = os.path.join(chothia_dir, file)
    renumber_pdb_wt_constant(filepath, outpath)

Chain A: V-domain 122개 잔기를 renumbering합니다...
Chain B: V-domain 106개 잔기를 renumbering합니다...
Renumbering 완료. 파일 저장: /home/psh/data/CAD6_GKR01I/pdb_chothia/gkr01i.pdb


In [13]:
###### Create Metadata ######
meta_dir = INPUT_PDB_DIR.replace('pdb', 'meta')
!python "/home/psh/protein-frame-flow/data/process_pdb_files.py" \
    --pdb_dir {INPUT_PDB_DIR} \
    --chothia_dir {chothia_dir} \
    --num_processes 8 \
    --write_dir {meta_dir} \
    --debug

Files will be written to /home/psh/data/CAD6_GKR01I/meta
/opt/anaconda3/envs/fm/lib/python3.10/site-packages/mdtraj/formats/pdb/pdbfile.py:200: UserWarning: Unlikely unit cell vectors detected in PDB file likely resulting from a dummy CRYST1 record. Discarding unit cell vectors.
  warnings.warn('Unlikely unit cell vectors detected in PDB file likely '
Finished /home/psh/data/CAD6_GKR01I/pdb/gkr01i.pdb in 1.17s
Finished processing 1/1 files


In [14]:
###### Process UNK tokens ######
import os
from data.utils import read_pkl, write_pkl

root_dir = INPUT_PDB_DIR.replace("/pdb", "/meta")

def process_one_pkl(pkl_path):
    data = read_pkl(pkl_path)

    # 필수 키 없으면 스킵
    if "aatype" not in data or "bb_mask" not in data:
        return

    aatype = data["aatype"]
    bb_mask = data["bb_mask"]

    # 길이 체크 (안 맞으면 스킵)
    if len(aatype) != len(bb_mask):
        return

    modified = False

    for i, aa in enumerate(aatype):
        if aa == 20 and bb_mask[i] == 1:
            bb_mask[i] = 0
            modified = True

    if modified:
        data["bb_mask"] = bb_mask
        write_pkl(pkl_path, data, create_dir=False)
    # 아무 변경 없으면 그냥 스킵


for root, _, files in os.walk(root_dir):
    for fname in files:
        if fname.endswith(".pkl"):
            pkl_path = os.path.join(root, fname)
            process_one_pkl(pkl_path)

In [15]:
###### Process UNK tokens ######

import os
from data.utils import read_pkl, write_pkl

root_dir = INPUT_PDB_DIR.replace("/pdb", "/meta")

def filter_by_bb_mask(pkl_path):
    data = read_pkl(pkl_path)

    if "bb_mask" not in data:
        return

    bb_mask = data["bb_mask"]

    L = len(bb_mask)

    # keep index
    keep_idx = [i for i, v in enumerate(bb_mask) if v == 1]

    # 전부 1이면 스킵
    if len(keep_idx) == L:
        return

    new_data = {}

    for k, v in data.items():

        # 리스트 / 튜플
        if isinstance(v, (list, tuple)):
            if len(v) == L:
                new_data[k] = [v[i] for i in keep_idx]
            else:
                new_data[k] = v

        # torch tensor
        elif hasattr(v, "shape"):
            try:
                if v.shape[0] == L:
                    new_data[k] = v[keep_idx]
                else:
                    new_data[k] = v
            except Exception:
                new_data[k] = v

        else:
            new_data[k] = v

    write_pkl(pkl_path, new_data, create_dir=False)


for root, _, files in os.walk(root_dir):
    for fname in files:
        if fname.endswith(".pkl"):
            filter_by_bb_mask(os.path.join(root, fname))


In [16]:
###### Process CDR definition ######

import pandas as pd
import pickle
import numpy as np
import os 
import glob 

########################################################################
# 1. 파일 경로 설정
root_dir = INPUT_PDB_DIR.replace("/pdb", "/meta")
pkl_path = glob.glob(os.path.join(root_dir, "**", "*.pkl"), recursive=True)[0]
csv_path = os.path.join(root_dir, 'metadata.csv')
new_csv_path = os.path.join(root_dir, 'metadata.csv')



########################################################################

# 2. 제공해주신 아미노산 매핑 정보
restypes = [
    'A', 'R', 'N', 'D', 'C', 'Q', 'E', 'G', 'H', 'I', 'L', 'K', 'M', 'F', 'P',
    'S', 'T', 'W', 'Y', 'V'
]

def get_sequence_from_aatype(aatype_data):
    """정수형 aatype을 문자열 서열로 변환"""
    if isinstance(aatype_data, np.ndarray):
        aatype_data = aatype_data.flatten()
    elif isinstance(aatype_data, list):
        aatype_data = np.array(aatype_data).flatten()

    seq_chars = []
    for idx in aatype_data:
        if 0 <= idx < len(restypes):
            seq_chars.append(restypes[idx])
        else:
            seq_chars.append('X')
            
    return "".join(seq_chars)

# 3. pkl 파일 로드 및 서열 변환
if os.path.exists(pkl_path):
    with open(pkl_path, 'rb') as f:
        data = pickle.load(f)
    
    aatype = data['aatype']
    full_sequence = get_sequence_from_aatype(aatype)
    
    print(f"Recovered Sequence (len={len(full_sequence)}): {full_sequence[:30]}...")
else:
    raise FileNotFoundError(f"{pkl_path} 파일이 존재하지 않습니다.")

# 4. CDR 위치 탐색
update_dict = {}
for cdr_name, seq in CDR_SEQS.items():
    start_idx = full_sequence.find(seq)
    
    if start_idx != -1:
        # Python slicing 기준으로 end 계산 후, 저장 시 -1 (요청하신 로직 유지)
        end_idx = start_idx + len(seq)
        
        update_dict[f"{cdr_name}_start"] = start_idx
        update_dict[f"{cdr_name}_end"] = end_idx - 1
        print(f"Found {cdr_name}: {start_idx} ~ {end_idx-1} (Length: {len(seq)})")
    else:
        print(f"Warning: {cdr_name} sequence not found in aatype sequence.")

# 5. CSV 업데이트 (모든 행 적용)
if os.path.exists(csv_path) and update_dict:
    try:
        df = pd.read_csv(csv_path)
        
        print(f"\nUpdating {len(df)} rows in the CSV...")

        # update_dict의 내용을 모든 행에 적용
        for col, val in update_dict.items():
            # 컬럼이 없으면 새로 생성, 있으면 덮어쓰기 (모든 행에 val 값 할당)
            df[col] = val
            
        df.to_csv(new_csv_path, index=False)
        print(f"Successfully updated metadata at: {new_csv_path}")
        
        # 결과 확인을 위해 첫 3행 출력
        print("\nUpdated DataFrame Sample (First 3 rows):")
        print(df[[k for k in update_dict.keys()]].head(3))
        
    except Exception as e:
        print(f"Error updating CSV: {e}")
else:
    print("CSV file not found or no CDRs matched.")

Recovered Sequence (len=555): EVQLVQSGAEVKKPGASVKVSCKASGYTFT...
Found h1: 25 ~ 34 (Length: 10)
Found h2: 49 ~ 56 (Length: 8)
Found h3: 98 ~ 110 (Length: 13)
Found l1: 145 ~ 155 (Length: 11)
Found l2: 171 ~ 177 (Length: 7)
Found l3: 210 ~ 217 (Length: 8)

Updating 1 rows in the CSV...
Successfully updated metadata at: /home/psh/data/CAD6_GKR01I/meta/metadata.csv

Updated DataFrame Sample (First 3 rows):
   h1_start  h1_end  h2_start  h2_end  h3_start  h3_end  l1_start  l1_end  \
0        25      34        49      56        98     110       145     155   

   l2_start  l2_end  l3_start  l3_end  
0       171     177       210     217  
